# 01 – Veri İnceleme

**CevapAnaliz** projesinin ilk not defteri. Model kurmadan önce elimizdeki veriyi tanıyacağız:

- Veri setlerinde ne var, bir kayıt neye benziyor?
- Kaç cevap ve kaç soru var?
- Puanlar nasıl dağılıyor? (Çoğu cevap iyi mi, kötü mü?)
- Cevaplar ne kadar uzun?
- Referans cevapla ortak kelime sayısı puanı ne kadar açıklıyor?

Bu soruların cevapları, hangi yöntemi seçeceğimizi ve başarıyı nasıl ölçeceğimizi belirler. Buradaki grafikler ve sayılar haftalık rapora da girecek.

### Nasıl çalıştırılır?
1. Üst menüden **Çalışma zamanı → Tümünü çalıştır** de (ya da her hücrede **Shift + Enter**).
2. İlk hücre Google Drive'a erişim izni isteyecek: açılan pencerede hesabını seçip **İzin ver** de.
3. Bu not defteri için GPU gerekmez.

## 1. Kurulum

Bu hücre dört iş yapar:
1. **Drive'ı bağlar.** Colab'daki bilgisayar geçicidir; sekmeyi kapatınca içindekiler silinir. Verileri ve grafikleri kalıcı olsun diye Drive'daki `CevapAnaliz` klasörüne kaydedeceğiz.
2. **Kodları GitHub'dan çeker.** Böylece ekipten biri kodu güncellediğinde herkes en son hâlini kullanır.
3. Projedeki `data` klasörünü **Drive'daki klasöre bağlar**; veriler bir kez indirilir, sonraki açılışlarda hazır olur.
4. Grafiklerin kaydedileceği `sekiller` klasörünü oluşturur.

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"
SEKIL.mkdir(parents=True, exist_ok=True)
print("Kodlar  :", PROJE)
print("Kayıtlar:", DRIVE)

## 2. Verileri indir ve ortak biçime çevir

Üç açık veri seti kullanıyoruz:

| Veri seti | İçerik |
|---|---|
| **Mohler** | Bir üniversitedeki giriş seviyesi programlama dersinin ödev ve sınav soruları |
| **SciEntsBank** (SemEval-2013) | Ortaokul düzeyi fen soruları |
| **Beetle** (SemEval-2013) | Elektrik devreleri öğreten bir eğitim yazılımının soruları |

Her veri setinin dosya biçimi farklı. `ortak_bicim.py` betiği hepsini **aynı sütunlara sahip tek bir tabloya** çevirir (3. hafta raporunda tasarladığımız biçim). Puanlar da 0–1 aralığına çevrilir:

- Mohler 0–5 arası puan kullanır → puan / 5 (örneğin 4 puan → 0,8)
- SemEval sayı yerine etiket kullanır → doğru = 1, kısmen doğru = 0,5, diğerleri = 0

İlk çalıştırmada veriler indirilir (birkaç saniye sürer); sonraki çalıştırmalarda Drive'dan okunur.

In [ ]:
for betik in ["scripts/veri_indir.py", "scripts/ortak_bicim.py"]:
    sonuc = subprocess.run([sys.executable, betik], capture_output=True, text=True)
    print(sonuc.stdout[-1500:], sonuc.stderr[-1500:])

## 3. Ortak tablo neye benziyor?

Her satır **bir öğrenci cevabıdır**. Sütunlar:

| Sütun | Anlamı |
|---|---|
| `veri_seti` | Kaydın geldiği veri seti |
| `bolum` | Veri setinin kendi eğitim/test ayrımı (Mohler'de yok, `tum` yazar) |
| `soru_id` | Sorunun numarası |
| `soru` | Soru metni |
| `referans_cevap` | Öğretmenin doğru kabul ettiği cevap |
| `ogrenci_cevap` | Öğrencinin cevabı |
| `orijinal_puan` | Veri setindeki puan ya da etiket |
| `puan` | 0–1 aralığına çevrilmiş puan — modelin tahmin etmeye çalışacağı değer |

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

df = pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False)
print(f"Toplam {len(df):,} cevap".replace(",", "."))
display(df.sample(5, random_state=0))

## 4. Her veri setinden bir örnek

Bir soru, öğretmenin referans cevabı, en yüksek ve en düşük puanlı öğrenci cevabı. Sistemimizin yapacağı iş tam olarak bu: öğrenci cevabını referans cevapla karşılaştırıp puan vermek.

In [ ]:
import textwrap

def yaz(baslik, metin):
    print(f"{baslik:<16}" + textwrap.fill(metin, 100, subsequent_indent=" " * 16))

for veri_seti in ["mohler", "scientsbank", "beetle"]:
    alt = df[df["veri_seti"] == veri_seti]
    soru_id = alt.groupby("soru_id")["puan"].std().idxmax()   # puanları en çeşitli soru
    soru = alt[alt["soru_id"] == soru_id].sort_values("puan")
    print("=" * 116)
    print(veri_seti.upper())
    yaz("Soru:", soru["soru"].iloc[0])
    yaz("Referans:", soru["referans_cevap"].iloc[0])
    yaz(f"İyi ({soru['orijinal_puan'].iloc[-1]}):", soru["ogrenci_cevap"].iloc[-1])
    yaz(f"Kötü ({soru['orijinal_puan'].iloc[0]}):", soru["ogrenci_cevap"].iloc[0])

## 5. Ne kadar veri var?

In [ ]:
from src.puanlama.ozellikler import kelimeler, ngram_ortusme

df["cevap_kelime"] = df["ogrenci_cevap"].map(lambda s: len(kelimeler(s)))
df["referans_kelime"] = df["referans_cevap"].map(lambda s: len(kelimeler(s)))

ozet = df.groupby("veri_seti").agg(
    cevap=("ogrenci_cevap", "size"),
    soru=("soru_id", "nunique"),
    cevap_kelime_ortanca=("cevap_kelime", "median"),
    referans_kelime_ortanca=("referans_kelime", "median"),
    ortalama_puan=("puan", "mean"),
).loc[["mohler", "scientsbank", "beetle"]]
ozet["soru_basina_cevap"] = (ozet["cevap"] / ozet["soru"]).round(1)
display(ozet.round(2))

- Toplamda **18.276 cevap**, **318 soru** var. En büyük veri seti SciEntsBank (10.804 cevap).
- Cevaplar **kısa**: ortanca cevap 9–16 kelime. Yani bu bir "kompozisyon puanlama" değil, **kısa cevap puanlama** problemi.
- Mohler'de soru başına yaklaşık 28 cevap varken Beetle'da 124 cevap var: Beetle'da az soruya çok cevap verilmiş.

> **Ortanca** (medyan): Değerler küçükten büyüğe sıralandığında ortada kalan değer. Ortalamadan farkı, birkaç çok uzun cevaptan etkilenmemesidir.

## 6. Puanlar nasıl dağılıyor?

Bu soru önemli: Eğer cevapların çoğu tam puan almışsa, "herkese yüksek puan ver" diyen aptal bir sistem bile ilk bakışta iyi görünebilir. Bunu bilmek, başarıyı **doğru ölçütle** ölçmemizi sağlar.

In [ ]:
import matplotlib.pyplot as plt

RENK = {"mohler": "#2a78d6", "scientsbank": "#eb6834", "beetle": "#1baf7a"}
AD = {"mohler": "Mohler", "scientsbank": "SciEntsBank", "beetle": "Beetle"}
ETIKET_TR = {"correct": "doğru", "partially_correct_incomplete": "kısmen\ndoğru",
             "contradictory": "çelişkili", "irrelevant": "alakasız", "non_domain": "alan\ndışı"}

def sade(ax):
    for kenar in ["top", "right"]:
        ax.spines[kenar].set_visible(False)
    ax.spines["left"].set_color("#b5b4ad"); ax.spines["bottom"].set_color("#b5b4ad")
    ax.tick_params(colors="#52514e")
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)

fig, eksenler = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, veri_seti in zip(eksenler, ["mohler", "scientsbank", "beetle"]):
    alt = df[df["veri_seti"] == veri_seti]
    if veri_seti == "mohler":
        sayim = alt["orijinal_puan"].astype(float).value_counts(normalize=True).sort_index()
        etiketler = [f"{p:g}".replace(".", ",") for p in sayim.index]
        ax.set_xlabel("Öğretmen puanı (0–5)")
    else:
        sira = ["correct", "partially_correct_incomplete", "contradictory", "irrelevant", "non_domain"]
        sayim = alt["orijinal_puan"].value_counts(normalize=True).reindex(sira)
        etiketler = [ETIKET_TR[e] for e in sira]
        ax.set_xlabel("Öğretmen etiketi")
    ax.bar(range(len(sayim)), sayim.values * 100, width=0.7, color=RENK[veri_seti])
    ax.set_xticks(range(len(sayim)), etiketler)
    ax.set_title(f"{AD[veri_seti]} ({len(alt):,} cevap)".replace(",", "."), loc="left", fontsize=12)
    ax.set_ylabel("Cevapların yüzdesi (%)")
    sade(ax)
fig.suptitle("Puan dağılımı", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "01_puan_dagilimi.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne görüyoruz?**

- **Mohler dengesiz:** Cevapların yaklaşık **yarısı (%49) tam puan** almış, **%72'si 4 ve üzeri** puanlı. 2 ve altı puan alan cevaplar sadece %7,5. Ortalama puan 5 üzerinden yaklaşık 4,2.
- **SciEntsBank ve Beetle daha dengeli:** Cevapların yaklaşık %41–42'si doğru, %23–24'ü kısmen doğru, geri kalan %35'i yanlış (çelişkili, alakasız veya alan dışı).

**Bu neden önemli?** Mohler'de her cevaba 4,2 puan veren bir sistem, ortalamada öğretmenden sadece 0,7 puan kadar sapar; yani "ortalama hata" ölçütüne bakarsak bu işe yaramaz sistem başarılı görünür. Bu yüzden başarıyı **korelasyon** ve **QWK** ile ölçüyoruz: Bu ölçütler, sistemin iyi cevabı kötü cevaptan **ayırt edip edemediğine** bakar. Herkese aynı puanı veren bir sistemin korelasyonu sıfırdır.

## 7. Cevaplar ne kadar uzun? Uzun cevap daha mı çok puan alıyor?

Puanlama motorumuzun üç özelliğinden biri **uzunluk oranı** (FR-03). Peki uzunluk puanla gerçekten ilişkili mi? Ayrıca adillik gereksinimimiz (NFR-04) kısa cevap veren öğrencinin haksız yere düşük puan almamasını istiyor; bu yüzden uzunluk–puan ilişkisini bilmemiz gerekiyor.

> **Spearman korelasyonu:** İki değişkenin birlikte artıp azalma eğilimini −1 ile 1 arasında bir sayıyla gösterir. 0'a yakınsa ilişki yok, 1'e yakınsa biri arttıkça diğeri de artıyor demektir.

In [ ]:
from scipy.stats import spearmanr

fig, eksenler = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, veri_seti in zip(eksenler, ["mohler", "scientsbank", "beetle"]):
    alt = df[df["veri_seti"] == veri_seti]
    ax.hist(alt["cevap_kelime"].clip(upper=60), bins=range(0, 62, 2), color=RENK[veri_seti], rwidth=0.85)
    ax.axvline(alt["cevap_kelime"].median(), color="#0b0b0b", linewidth=1, linestyle="--")
    ax.text(alt["cevap_kelime"].median() + 1, ax.get_ylim()[1] * 0.92,
            f"ortanca: {alt['cevap_kelime'].median():g} kelime", fontsize=9, color="#0b0b0b")
    r = spearmanr(alt["cevap_kelime"], alt["puan"])[0]
    ax.set_title(f"{AD[veri_seti]} · uzunluk–puan Spearman: {r:.2f}".replace(".", ","), loc="left", fontsize=12)
    ax.set_xlabel("Öğrenci cevabının kelime sayısı (60+ birleştirildi)")
    ax.set_ylabel("Cevap sayısı")
    sade(ax)
fig.suptitle("Cevap uzunlukları", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "01_cevap_uzunlugu.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne görüyoruz?**

- Cevapların çoğu **20 kelimenin altında**.
- Uzunluk ile puan arasındaki ilişki **zayıf**: Mohler'de neredeyse yok (0,03), SciEntsBank'te çok zayıf (0,09), Beetle'da zayıf (0,27).
- Yani **uzun yazmak tek başına yüksek puan getirmiyor**. Bu, adillik açısından iyi bir haber: Öğretmenler uzunluğa değil içeriğe puan vermiş. Bizim sistemimiz de uzunluğa fazla güvenmemeli; uzunluk oranı ancak diğer özelliklerle birlikte yardımcı olabilir.

## 8. Referansla ortak kelime oranı puanı ne kadar açıklıyor?

**N-gram örtüşmesi**, referans cevaptaki kelimelerin ve ikili kelime gruplarının (örneğin "software product") ne kadarının öğrenci cevabında da geçtiğini gösterir: 0 = hiç ortak kelime yok, 1 = referanstaki her şey öğrenci cevabında var.

Kelime tabanlı yöntemlerin (TF-IDF gibi) temel varsayımı şudur: *Doğru cevap, referansla aynı kelimeleri kullanır.* Bu varsayımın ne kadar doğru olduğuna bakalım.

In [ ]:
df["ortusme"] = [ngram_ortusme(r, c) for r, c in zip(df["referans_cevap"], df["ogrenci_cevap"])]

def puan_grubu(satir):
    if satir["veri_seti"] == "mohler":
        p = float(satir["orijinal_puan"])
        return "0–2" if p <= 2 else "2,5–3,5" if p <= 3.5 else "4–4,5" if p <= 4.5 else "5"
    return {0.0: "yanlış", 0.5: "kısmen doğru", 1.0: "doğru"}[satir["puan"]]

df["puan_grubu"] = df.apply(puan_grubu, axis=1)
SIRA = {"mohler": ["0–2", "2,5–3,5", "4–4,5", "5"], "scientsbank": ["yanlış", "kısmen doğru", "doğru"]}
SIRA["beetle"] = SIRA["scientsbank"]

fig, eksenler = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, veri_seti in zip(eksenler, ["mohler", "scientsbank", "beetle"]):
    alt = df[df["veri_seti"] == veri_seti]
    gruplar = [alt.loc[alt["puan_grubu"] == g, "ortusme"] for g in SIRA[veri_seti]]
    kutu = ax.boxplot(gruplar, widths=0.55, patch_artist=True, showfliers=False,
                      medianprops={"color": "#0b0b0b", "linewidth": 1.5})
    for parca in kutu["boxes"]:
        parca.set_facecolor(RENK[veri_seti]); parca.set_alpha(0.8); parca.set_edgecolor(RENK[veri_seti])
    ax.set_xticks(range(1, len(gruplar) + 1), SIRA[veri_seti])
    r = spearmanr(alt["ortusme"], alt["puan"])[0]
    ax.set_title(f"{AD[veri_seti]} · Spearman: {r:.2f}".replace(".", ","), loc="left", fontsize=12)
    ax.set_xlabel("Öğretmen puanı" if veri_seti == "mohler" else "Öğretmen etiketi")
    sade(ax)
eksenler[0].set_ylabel("Referansla n-gram örtüşmesi (0–1)")
fig.suptitle("Ortak kelime oranı ve puan", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "01_ortusme_puan.png", dpi=150, bbox_inches="tight")
plt.show()

**Grafik nasıl okunur?** Her kutu bir puan grubundaki cevapları gösterir. Kutunun ortasındaki siyah çizgi ortancadır; kutu, cevapların ortadaki yarısını kapsar.

**Ne görüyoruz?**

- Puan arttıkça kutular yukarı çıkıyor: **Yüksek puanlı cevaplar referansla daha çok kelime paylaşıyor.** Spearman değerleri 0,39–0,55 arasında, yani orta düzey bir ilişki var. Kelime tabanlı yöntemin bir işe yaramasının sebebi bu.
- Ama kutular **birbirinin üstüne biniyor**: Aynı örtüşme değerine sahip bir cevap tam puan da almış olabilir, düşük puan da. Ortak kelimeye bakmak **tek başına yetmiyor**.

## 9. Kelimeler farklı, anlam aynı

Kelime tabanlı yöntemin zayıf noktasını somut olarak görelim: Mohler'de **tam puan (5/5) almış** ama referansla **en az kelime paylaşan** cevaplar.

In [ ]:
tam = df[(df["veri_seti"] == "mohler") & (df["puan"] == 1.0) & (df["cevap_kelime"] >= 6)]
for _, satir in tam.nsmallest(3, "ortusme").iterrows():
    print("=" * 116)
    yaz("Soru:", satir["soru"])
    yaz("Referans:", satir["referans_cevap"])
    yaz("Öğrenci (5/5):", satir["ogrenci_cevap"])
    print(f"{'Örtüşme:':<16}{satir['ortusme']:.2f}".replace(".", ","))

Bu cevaplar öğretmenden tam puan almış, ama referansla neredeyse hiç ortak kelimeleri yok; öğrenci aynı fikri **kendi kelimeleriyle** anlatmış. Kelime sayan bir yöntem bu cevaplara düşük puan verir. İşte bu yüzden **anlam benzerliğine** bakan yapay zekâ modellerine (Sentence-BERT) ihtiyacımız var.

## Özet: Bu not defterinden çıkanlar

| Bulgu | Sonucu |
|---|---|
| 3 veri seti, 18.276 cevap, 318 soru | Model eğitmek ve test etmek için yeterli veri var |
| Cevaplar kısa (ortanca 9–16 kelime) | Problem bir *kısa cevap puanlama* problemi |
| Mohler'de cevapların %72'si 4 ve üzeri puanlı | Başarıyı ortalama hatayla değil, **korelasyon ve QWK** ile ölçmeliyiz |
| Uzunluk ile puan arasında zayıf ilişki (0,03–0,27) | Uzunluk tek başına iyi bir özellik değil; öğretmenler içeriğe puan vermiş |
| Ortak kelime ile puan arasında orta ilişki (0,39–0,55) | Kelime tabanlı yöntem kısmen işe yarar ama yetmez |
| Tam puanlı cevapların bir kısmı referansla çok az kelime paylaşıyor | **Anlam benzerliğine** bakan modellere ihtiyaç var |

Grafikler Drive'daki `CevapAnaliz/sekiller` klasörüne kaydedildi; rapora oradan eklenebilir.

**Sonraki not defteri: 02 – Ön İşleme.** Metin temizleme adımlarının (gereksiz kelimeleri silme, kelimeyi köküne indirme) kelime tabanlı yöntemi iyileştirip iyileştirmediğini deneyeceğiz.